In [2]:
# In kaggle v33026.ipynb, I only included the audio files from train_soundscapes.
# In this file, I also include the audio files from train_audio.

import librosa
import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [26]:
train_soundscapes_labels_df = pd.read_csv("train_soundscapes_labels.csv")

train_soundscapes_reduced_df = pd.DataFrame({
    "filename": train_soundscapes_labels_df.iloc[:, :2].astype(str).agg(''.join, axis=1),
    "primary_label": train_soundscapes_labels_df.iloc[:, 3]
})

In [9]:
train_soundscapes_labels_df.head(5)

,filename,start,end,primary_label
0,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:00,00:00:05,22961;23158;24321;517063;65380
1,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:05,00:00:10,22961;23158;24321;517063;65380
2,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:10,00:00:15,22961;23158;24321;517063;65380
3,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:15,00:00:20,22961;23158;24321;517063;65380
4,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:20,00:00:25,22961;23158;24321;517063;65380


In [10]:
train_soundscapes_reduced_df.head(5)

,filename,primary_label
0,BC2026_Train_0039_S22_20211231_201500.ogg00:00:00,22961;23158;24321;517063;65380
1,BC2026_Train_0039_S22_20211231_201500.ogg00:00:05,22961;23158;24321;517063;65380
2,BC2026_Train_0039_S22_20211231_201500.ogg00:00:10,22961;23158;24321;517063;65380
3,BC2026_Train_0039_S22_20211231_201500.ogg00:00:15,22961;23158;24321;517063;65380
4,BC2026_Train_0039_S22_20211231_201500.ogg00:00:20,22961;23158;24321;517063;65380


In [11]:
len(train_soundscapes_reduced_df)

1478

In [27]:
taxonomy_df = pd.read_csv("taxonomy.csv")
classes = ['Insecta', 'Reptilia', 'Amphibia', 'Mammalia', 'Aves']
filenames = {value: [] for value in classes}

for entry in train_soundscapes_reduced_df.itertuples():
    filename = entry.filename
    primary_label = entry.primary_label.split(';')

    found = {value: False for value in classes}
    
    for label in primary_label:
        for value in classes:
            if taxonomy_df[taxonomy_df['primary_label'] == label]['class_name'].values[0] == value:
                found[value] = True
    
    for value in classes:
         if found[value]:
            filenames[value].append(filename)

In [28]:
# train_audio has 206 folders, each folder corresponds to a primary label.

folder_path = "train_audio"

num_folders = sum(
    os.path.isdir(os.path.join(folder_path, item))
    for item in os.listdir(folder_path)
)

print(num_folders)

206


In [29]:
# train_audio has 206 folders, each folder corresponds to a primary label.
# The files in train_soundscapes_reduced_df are all 5 seconds, but the ones in train_audio vary in length.
# This block classifies all of the sound files in train_audio_df.csv
# When we make a model, we will split these files into 5 second intervals.

# One issue is that there are 235 species.

# This code takes 10 minutes to run.

from time import time

start_time = time()

train_audio_df = pd.DataFrame(columns=['filename', 'primary_label'])

item_count = 0

for item in os.listdir(folder_path):
    item_count += 1
    if os.path.isdir(os.path.join(folder_path, item)):
        primary_label = taxonomy_df[taxonomy_df['primary_label'] == item]['class_name'].values[0]
        for file in os.listdir(os.path.join(folder_path, item)):
            if file.endswith('.ogg'):
                
                filename = os.path.join(item, file)

                audio, sr = librosa.load(os.path.join(folder_path, item, file), sr=None)
                duration = librosa.get_duration(y=audio, sr=sr)
                
                if duration >= 5:
                    # Add filename and class to file.
                    train_audio_df.loc[len(train_audio_df)] = [filename, primary_label]

    print(f"Time taken: {time() - start_time} seconds")
    print(f"Items processed: {item_count}")

Time taken: 2.260925054550171 seconds
Items processed: 1
Time taken: 8.082215070724487 seconds
Items processed: 2
Time taken: 15.178282022476196 seconds
Items processed: 3
Time taken: 15.702569007873535 seconds
Items processed: 4
Time taken: 18.61775302886963 seconds
Items processed: 5
Time taken: 18.69682288169861 seconds
Items processed: 6
Time taken: 19.70902991294861 seconds
Items processed: 7
Time taken: 19.73676300048828 seconds
Items processed: 8
Time taken: 29.770681858062744 seconds
Items processed: 9
Time taken: 36.55143213272095 seconds
Items processed: 10
Time taken: 39.93571209907532 seconds
Items processed: 11
Time taken: 39.967047929763794 seconds
Items processed: 12
Time taken: 43.087323904037476 seconds
Items processed: 13
Time taken: 43.1636381149292 seconds
Items processed: 14
Time taken: 43.39476799964905 seconds
Items processed: 15
Time taken: 43.97493600845337 seconds
Items processed: 16
Time taken: 44.38703489303589 seconds
Items processed: 17
Time taken: 46.2964

In [30]:
len(train_audio_df)

32948

In [ ]:
# Because this dataframe is very large, we save it as a csv file
# so that we can load it later without having to run the above code again.

train_audio_df.to_csv("train_audio_df.csv", index=False)

In [17]:
# Loading the folder

train_audio_df = pd.read_csv("train_audio_df.csv")

In [18]:
train_audio_df.head(5)

,filename,primary_label
0,crebec1/XC119358.ogg,Aves
1,crebec1/XC602873.ogg,Aves
2,crebec1/XC844032.ogg,Aves
3,crebec1/XC611358.ogg,Aves
4,crebec1/XC995496.ogg,Aves


In [31]:
NUM_CLASSES = 5
SR = 32000
N_MELS = 128
X_multi = []
y_multi = []

# This loop converts each audio file in train_soundscapes into a mel spectrogram.

for filename in train_soundscapes_reduced_df['filename']:
    file = filename[:-8]
    start = int(filename[-2:])
    
    audio, _ = librosa.load(
        f"train_soundscapes/{file}",
        sr=SR,
        offset=start,
        duration=5
    )

# Converts the audio to a mel spectrogram, and normalizes it to the range [0, 1].

    mel = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())

    X_multi.append(mel_db[..., np.newaxis])

    # 🔥 MULTI-LABEL TARGET
    label_vector = [0] * NUM_CLASSES
    for i, cls in enumerate(classes):
        if filename in filenames[cls]:
            label_vector[i] = 1

    y_multi.append(label_vector)
    
X_multi = np.array(X_multi, dtype=np.float32)
y_multi = np.array(y_multi, dtype=np.float32)

# X is the list of spectrograms, and y is the list of multi-label vectors.

In [ ]:
# We repeat the same process for the audio files in train_audio_df.
# This code is somewhat inefficient, but we only have to run it once.
# It takes 13 minutes to run this code.

from time import time

count = 0
X_single = []
y_single = []
t = time()

EXPECTED_FRAMES = int(np.ceil((SR * 5) / 512))

for row in train_audio_df.itertuples():
    count += 1
    if count%100 == 0:
        print(count, time() - t)
    filename = row.filename
    
    audio, _ = librosa.load(
        f"train_audio/{filename}",
        sr=SR
    )

# After breaking the audio files into 5 second chunks, there were far too many files.
# Rather than processing every file, we take a few chunks.
# If the file is less than 15 seconds, we take 1 chunk. Otherwise, we take 2.

    chunk_length = 5 * SR

    if len(audio) < 2 * chunk_length:
        # 5–10 sec → just take first chunk
        audio_clips = [audio[:chunk_length]]

    elif len(audio) < 3 * chunk_length:
        # 10–15 sec → allow overlap (random is fine)
        audio_clips = []
        for _ in range(2):
            start = np.random.randint(0, len(audio) - chunk_length)
            audio_clips.append(audio[start:start + chunk_length])

    else:
        # ≥15 sec → enforce separation safely
        start1 = np.random.randint(0, len(audio) - chunk_length)
        start2 = np.random.randint(0, len(audio) - chunk_length)

        # Try a few times only (avoid slow loops)
        for _ in range(5):
            if abs(start1 - start2) >= chunk_length:
                break
            start2 = np.random.randint(0, len(audio) - chunk_length)

        audio_clips = [audio[start1:start1 + chunk_length], audio[start2:start2 + chunk_length]]

# Converts the audio files in audio_clips to a mel spectrogram, and normalizes it to the range [0, 1].

    for audio_clip in audio_clips:

        mel = librosa.feature.melspectrogram(
            y=audio_clip,
            sr=SR,
            n_mels=N_MELS
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())

        # Pad / trim
        if mel_db.shape[1] < EXPECTED_FRAMES:
            mel_db = np.pad(mel_db, ((0, 0), (0, EXPECTED_FRAMES - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :EXPECTED_FRAMES]

        X_single.append(mel_db[..., np.newaxis])

        label_vector = [0] * NUM_CLASSES
        for j, cls in enumerate(classes):
            if row.primary_label == cls:
                label_vector[j] = 1

        y_single.append(label_vector)

X_single = np.array(X_single, dtype=np.float32)
y_single = np.array(y_single, dtype=np.float32)

100 2.483482837677002
200 4.973445892333984
300 7.491352081298828
400 9.754539966583252
500 11.502514839172363
600 13.653581857681274
700 16.47809386253357
800 18.275458097457886
900 20.241528749465942
1000 22.42976999282837
1100 24.530609846115112
1200 26.498512983322144
1300 28.753445863723755
1400 31.197897911071777
1500 35.06235074996948
1600 37.5367329120636
1700 40.37897300720215
1800 43.171833753585815
1900 46.17383289337158
2000 48.12741994857788
2100 50.471137046813965
2200 52.6482298374176
2300 54.901602029800415
2400 56.583648920059204
2500 58.24816107749939
2600 61.24835515022278
2700 65.6317389011383
2800 71.61408805847168
2900 74.66432094573975
3000 77.37501096725464
3100 79.33660197257996
3200 81.8761260509491
3300 84.53369975090027
3400 86.54063892364502
3500 89.29617094993591
3600 92.05621409416199
3700 94.75500082969666
3800 96.40390300750732
3900 98.47418808937073
4000 100.47772288322449
4100 102.47477602958679
4200 104.51208186149597
4300 106.41469502449036
4400 108

In [22]:
# X contains 61,994 spectograms.
len(X)

61994

In [34]:
# Because it takes so long to generate the X and y arrays, we save them for later.

np.save("X_multi.npy", X_multi)
np.save("y_multi.npy", y_multi)
np.save("X_single.npy", X_single)
np.save("y_single.npy", y_single)

In [ ]:
# To load these arrays later, use the following code.

# IMPORTANT: If you have run the previous blocks in the past, you can start here after importing the packages from Block 1.

X_multi = np.load("X_multi.npy")
y_multi = np.load("y_multi.npy")
X_single = np.load("X_single.npy")
y_single = np.load("y_single.npy")

In [38]:
X = np.concatenate([X_multi, X_single], axis=0)
y = np.concatenate([y_multi, y_single], axis=0)

In [39]:
# We shuffle the elements of X and y so that we do not over-sample certain types of data.

indices = np.arange(len(X))
np.random.shuffle(indices)

X = X[indices]
y = y[indices]

np.save("X.npy", X)
np.save("y.npy", y)

In [40]:
print("Max labels:", y.sum(axis=1).max())
print("Multi-label count:", np.sum(y.sum(axis=1) > 1))

Max labels: 3.0
Multi-label count: 960


In [42]:
# Now that we have all the files, we can make a model.
# In a previous version of the code, we showed that the class weights of each class in our model are as follows.

class_weights = {'Insecta' : 0.8534296, 'Reptilia' : 11.257143, 'Amphibia' : 0.26092714, 'Mammalia' : 4.075862, 'Aves' : 0.7141994}
class_weights_array = np.array([class_weights[cls] for cls in class_weights])

In [43]:
def weighted_bce(y_true, y_pred):
    bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)

    weights = y_true * class_weights_array + (1 - y_true)

    return tf.reduce_mean(bce * weights, axis=-1)

In [45]:
# Performs a train-test split.

N = X.shape[0]

# Split at 80%
split = int(0.8 * N)
train_idx = indices[:split]
test_idx = indices[split:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [47]:
# We have far more single class data than multi-class data. An unweighted algorithm will say that a spectrogram is single class far too often.
# This block weights the train and test sets so that the model gets enough multi-class data.

multi_idx = np.where(y_train.sum(axis=1) > 1)[0]
single_idx = np.where(y_train.sum(axis=1) == 1)[0]

# repeat multi-label samples
# mild oversampling
oversampled_idx = np.concatenate([
    single_idx,
    np.repeat(multi_idx, 3)
])

# plus weighting
sample_weights = np.where(y_train.sum(axis=1) > 1, 2.0, 1.0)

np.random.shuffle(oversampled_idx)

X_train = X_train[oversampled_idx]
y_train = y_train[oversampled_idx]

In [ ]:
np.save("X_train.npy", X_train)
np.save("X_test.npy", X_test)
np.save("y_train.npy", y_train)
np.save("y_test.npy", y_test)

In [3]:
# If you have run all of the previous code before, you can start here after running Block 1.

X = np.load("X.npy")
X_train = np.load("X_train.npy")
X_test = np.load("X_test.npy")
y = np.load("y.npy")
y_train = np.load("y_train.npy")
y_test = np.load("y_test.npy")

In [4]:
print(X.shape)        # should be (N, 128, frames, 1)
print(y.shape)        # should be (N, NUM_CLASSES)
print(y.sum(axis=1))  # check how many multi-label samples

(63472, 128, 313, 1)
(63472, 5)
[1. 1. 1. ... 1. 1. 1.]


In [5]:
print("Max labels per sample:", y.sum(axis=1).max())
print("Number of multi-label samples:", np.sum(y.sum(axis=1) > 1))

Max labels per sample: 3.0
Number of multi-label samples: 960


In [6]:
print(len(y.sum(axis=1)))

63472


In [9]:
input_shape = X.shape[1:]  # (128, frames, 1)

NUM_CLASSES = 5

model = models.Sequential([
    layers.Input(shape=input_shape),
    
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(NUM_CLASSES, activation='sigmoid')  # 🔥 multi-label
])

In [10]:
def weighted_bce(y_true, y_pred):
    bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
    weights = y_true * class_weights_array + (1 - y_true)
    return tf.reduce_mean(bce * weights, axis=-1)

In [11]:
# Compiles the model.

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=weighted_bce,
    metrics=[
    tf.keras.metrics.AUC(multi_label=True),
    tf.keras.metrics.Precision(thresholds=0.3),
    tf.keras.metrics.Recall(thresholds=0.3)
]
)

In [12]:
# Sanity check

class_weights = {'Insecta' : 0.8534296, 'Reptilia' : 11.257143, 'Amphibia' : 0.26092714, 'Mammalia' : 4.075862, 'Aves' : 0.7141994}
class_weights_array = np.array([class_weights[cls] for cls in class_weights])
class_weights_array.shape == (NUM_CLASSES,)

True

In [13]:
sample_weights = np.where(y_train.sum(axis=1) > 1, 2.0, 1.0)
len(sample_weights) == len(X_train)

True

In [ ]:
# Trains the model
# This block took 3 hours on a CPU. The model had 97.5% precision and 98.6% recall.

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=64,
    shuffle=True,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )
    ]
)

Epoch 1/15
818/818 ━━━━━━━━━━━━━━━━━━━━ 920s 1s/step - auc: 0.8240 - loss: 0.1884 - precision: 0.6472 - recall: 0.9371 - val_auc: 0.6231 - val_loss: 0.1391 - val_precision: 0.9461 - val_recall: 0.9391
Epoch 2/15
818/818 ━━━━━━━━━━━━━━━━━━━━ 896s 1s/step - auc: 0.9212 - loss: 0.0713 - precision: 0.9285 - recall: 0.9606 - val_auc: 0.8049 - val_loss: 0.0734 - val_precision: 0.9519 - val_recall: 0.9508
Epoch 3/15
818/818 ━━━━━━━━━━━━━━━━━━━━ 716s 875ms/step - auc: 0.9420 - loss: 0.0594 - precision: 0.9408 - recall: 0.9658 - val_auc: 0.9103 - val_loss: 0.0471 - val_precision: 0.9651 - val_recall: 0.9738
Epoch 4/15
818/818 ━━━━━━━━━━━━━━━━━━━━ 745s 911ms/step - auc: 0.9532 - loss: 0.0513 - precision: 0.9508 - recall: 0.9714 - val_auc: 0.8845 - val_loss: 0.0715 - val_precision: 0.9531 - val_recall: 0.9622
Epoch 5/15
818/818 ━━━━━━━━━━━━━━━━━━━━ 734s 897ms/step - auc: 0.9614 - loss: 0.0456 - precision: 0.9558 - recall: 0.9736 - val_auc: 0.6806 - val_loss: 0.1249 - val_precision: 0.9419 - val_r

In [ ]:
# All of the thresholds performed well, but t = 0.5 performed the best at 0.99138.

y_pred = model.predict(X_test)

# try different thresholds
for t in [0.2, 0.3, 0.4, 0.5]:
    y_bin = (y_pred > t).astype(int)
    print(t, np.mean(y_bin == y_test))

397/397 ━━━━━━━━━━━━━━━━━━━━ 31s 79ms/step
0.2 0.9907837731390311
0.3 0.9911461205198897
0.4 0.991366679795195
0.5 0.9913824340291454


In [16]:
# We save the model. This version replaces the one in kaggle v33026.ipynb.

model.save("best_multilabel_model.keras")

In [ ]:
# This block shows how to load the model for future use.

from tensorflow.keras.models import load_model

model = load_model(
    "best_multilabel_model.keras",
    custom_objects={
        "loss": weighted_binary_crossentropy(class_weights)
    }
)